# Experimental Prediction Package for BEC Analog Hawking Radiation

**Phase 4 Wave 1 — Item 1A: Experimental Prediction Tables (Technical Notebook)**

---

## Introduction

This notebook synthesizes results from Papers 1–4 into platform-specific, publication-ready prediction tables for three BEC experiments:

| Platform | Species | Key Advantage |
|----------|---------|---------------|
| Steinhauer | ⁸⁷Rb | Proven apparatus, largest dataset |
| Heidelberg | ³⁹K | Feshbach tuning of $a_s$, $\kappa$-scanning |
| Trento | ²³Na | Spin-sonic two-sound system |

The predictions include:
1. Spectral occupation $n(\omega)$ at 5 representative frequencies per platform
2. Correction decomposition: dispersive ($\delta_{\text{disp}}$), dissipative ($\delta_{\text{diss}}$), noise floor ($n_{\text{noise}}$)
3. Required detector sensitivity to distinguish corrections from Planckian
4. Frequency range where exact WKB deviates $>1\%$ from perturbative EFT
5. $\kappa$-scaling test predictions (Heidelberg)
6. Platform comparison: which platform is best for which measurement

All physics is imported from `src.experimental.predictions` and `src.wkb.spectrum`. No formulas are reimplemented.

## Setup: Imports and Platform Parameters

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.core.visualizations import (
    COLORS,
    LAYOUT_TEMPLATE,
    FONT,
    TITLE_FONT,
    apply_layout,
)
from src.experimental.predictions import (
    compute_prediction_table,
    compute_all_predictions,
    compute_detector_requirements,
    compare_platforms,
    measurement_strategy,
    kappa_scaling_prediction,
    REPRESENTATIVE_OMEGAS,
)
from src.wkb.spectrum import (
    steinhauer_platform,
    heidelberg_platform,
    trento_platform,
    compute_spectrum,
    spectrum_summary,
    compare_exact_vs_perturbative,
)

platforms = {
    "Steinhauer": steinhauer_platform(),
    "Heidelberg": heidelberg_platform(),
    "Trento": trento_platform(),
}

PLATFORM_COLORS = {
    "Steinhauer": COLORS["Steinhauer"],
    "Heidelberg": COLORS["Heidelberg"],
    "Trento": COLORS["Trento"],
}

## 1. Prediction Tables

At each of the 5 representative frequencies $\omega/T_H \in \{0.5, 1.0, 2.0, 3.0, 5.0\}$, we compute:

- Total occupation number $n_{\text{total}} = n_{\text{Planck}} + \delta n_{\text{disp}} + \delta n_{\text{diss}} + n_{\text{noise}}$
- Fractional deviation from Planckian: $(n_{\text{total}} - n_{\text{Planck}})/n_{\text{Planck}}$
- Dominant correction at each frequency
- Signal-to-noise ratio per experimental shot

In [ ]:
# Compute prediction tables for all three platforms
tables = compute_all_predictions()

# Display as formatted dataframes
import pandas as pd

for name, table in tables.items():
    rows = []
    for p in table.predictions:
        rows.append({
            "omega/T_H": p.omega_over_T_H,
            "n_total": f"{p.n_total:.4f}",
            "n_Planck": f"{p.n_planck:.4f}",
            "n_noise": f"{p.n_noise:.2e}",
            "delta_disp": f"{p.delta_disp:.2e}",
            "delta_diss": f"{p.delta_diss:.2e}",
            "delta_k": f"{p.delta_k:.2e}",
            "frac_dev (%)": f"{p.fractional_deviation * 100:.4f}",
            "dominant": p.dominant_correction,
            "SNR/shot": f"{p.snr_per_shot:.3f}",
        })
    df = pd.DataFrame(rows)
    display(pd.DataFrame({"Platform": [name], "Atom": [table.atom],
                          "T_H": [f"{table.T_H:.4f}"],
                          "omega_max/T_H": [f"{table.omega_max_over_T_H:.2f}"]}))
    display(df)

The prediction tables above are the core deliverable for experimental collaborators. Key observations:

- At low frequencies ($\omega/T_H < 1$), all platforms are dominated by the dissipative correction $\delta_{\text{diss}}$
- At high frequencies ($\omega/T_H > 3$), the noise floor $n_{\text{noise}}$ becomes the dominant effect
- The dispersive correction $\delta_{\text{disp}} = -(\pi/6) D^2$ is frequency-independent and enters as a constant offset

## 2. Spectral Deviation Comparison

The fractional deviation from the Planckian spectrum is the primary observable. Positive deviations indicate excess phonon production (from dissipative heating); negative deviations indicate suppression (from dispersive blue-shifting of the effective temperature).

In [ ]:
# viz-ref: fig35

fig = go.Figure()

for name, table in tables.items():
    omegas = [p.omega_over_T_H for p in table.predictions]
    deviations = [p.fractional_deviation * 100 for p in table.predictions]

    fig.add_trace(go.Scatter(
        x=omegas,
        y=deviations,
        mode="lines+markers",
        name=name,
        line=dict(color=PLATFORM_COLORS[name], width=2),
        marker=dict(size=8),
    ))

fig.add_hline(y=0, line_dash="dot", line_color="grey", opacity=0.5)

apply_layout(fig,
    title="Fractional Deviation from Planckian Spectrum",
    xaxis_title="\u03c9 / T<sub>H</sub>",
    yaxis_title="Fractional deviation (%)",
)
fig.show()

## 3. Detector Sensitivity Requirements

For each platform, we compute the detector requirements for three measurement goals:

1. **Detect $\delta_{\text{diss}}$** at $3\sigma$ — the first-order dissipative correction
2. **Detect $n_{\text{noise}}$** at $3\sigma$ — the FDR noise floor (Paper 4 prediction)
3. **Distinguish exact WKB from perturbative EFT** at $>1\%$ deviation

The feasibility assessment uses Steinhauer's 2019 dataset (7000 shots, ~30% per-bin precision) as the current state of the art.

In [ ]:
# Detector requirements for all platforms
all_reqs = {}
for name, platform in platforms.items():
    all_reqs[name] = compute_detector_requirements(platform)

rows = []
for name, reqs in all_reqs.items():
    for req in reqs:
        rows.append({
            "Platform": name,
            "Goal": req.goal[:50],
            "Precision": f"{req.required_precision:.2e}",
            "Shots": f"{req.required_shots:,}",
            "Time (hrs)": f"{req.required_time_hours:.1f}",
            "Freq range": f"({req.frequency_range[0]:.1f}, {req.frequency_range[1]:.1f}) T_H",
            "Feasible": req.feasible,
            "Limiting factor": req.limiting_factor,
        })

display(pd.DataFrame(rows))

## 4. Required Shots Comparison

The bar chart below compares the number of experimental shots required for $3\sigma$ detection of the dissipative correction across all three platforms.

In [ ]:
# viz-ref: fig36

fig = go.Figure()

goals = ["delta_diss", "n_noise", "WKB vs EFT"]
goal_labels = ["Dissipative corr.", "Noise floor", "Exact vs pert."]

for i, (name, reqs) in enumerate(all_reqs.items()):
    shots = [req.required_shots for req in reqs]
    fig.add_trace(go.Bar(
        x=goal_labels,
        y=shots,
        name=name,
        marker_color=PLATFORM_COLORS[name],
    ))

fig.update_layout(
    barmode="group",
    yaxis_type="log",
)
apply_layout(fig,
    title="Required Shots for 3\u03c3 Detection",
    xaxis_title="Measurement Goal",
    yaxis_title="Number of shots",
)

# Reference line for current state of the art
fig.add_hline(y=7000, line_dash="dash", line_color="grey",
              annotation_text="Steinhauer 2019 (7000 shots)",
              annotation_position="top right")
fig.show()

## 5. $\kappa$-Scaling Prediction (Heidelberg K-39)

The Heidelberg platform can tune the scattering length $a_s$ via Feshbach resonance, which changes the healing length $\xi$ and surface gravity $\kappa$. The SK-EFT prediction is:

$$\frac{T_{\text{eff}}(\kappa)}{T_H(\kappa)} = 1 + \delta_{\text{disp}}(\kappa) + \delta_{\text{diss}}(\kappa)$$

The key discriminant:
- **Dispersive correction**: $\delta_{\text{disp}} \propto \kappa^2$ (quadratic scaling)
- **Dissipative correction**: $\delta_{\text{diss}} \propto \kappa^0$ (approximately constant)

This different scaling behavior provides an experimental handle to separate the two types of corrections by varying the condensate parameters.

In [ ]:
# viz-ref: fig37

kappa_pred = kappa_scaling_prediction(n_points=20)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Dispersive: \u03b1 = {kappa_pred.scaling_exponent_disp:.2f} (expect 2.0)",
        f"Dissipative: \u03b1 = {kappa_pred.scaling_exponent_diss:.2f} (expect 0.0)",
    ],
)

D_values = np.linspace(0.01, 0.05, 20)

fig.add_trace(go.Scatter(
    x=D_values,
    y=np.abs(kappa_pred.delta_disp_values),
    mode="lines+markers",
    name="|\u03b4_disp|",
    line=dict(color=COLORS["Steinhauer"], width=2),
    marker=dict(size=6),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=D_values,
    y=np.abs(kappa_pred.delta_diss_values),
    mode="lines+markers",
    name="|\u03b4_diss|",
    line=dict(color=COLORS["Heidelberg"], width=2),
    marker=dict(size=6),
), row=1, col=2)

fig.update_xaxes(title_text="Adiabaticity D", row=1, col=1)
fig.update_xaxes(title_text="Adiabaticity D", row=1, col=2)
fig.update_yaxes(title_text="|\u03b4|", type="log", row=1, col=1)
fig.update_yaxes(title_text="|\u03b4|", type="log", row=1, col=2)

apply_layout(fig,
    title="\u03ba-Scaling Test: Dispersive vs Dissipative",
)
fig.show()

The left panel confirms $\delta_{\text{disp}} \propto D^2 \propto \kappa^2$ (quadratic growth). The right panel shows $\delta_{\text{diss}}$ is approximately constant in $D$, confirming the prediction that dissipative corrections do not scale with surface gravity at leading order.

This test requires the Heidelberg group to perform multiple runs at different $a_s$ values and compare the effective temperature ratios. The different scaling laws provide a clean separation of the two correction types.

## 6. Noise Floor Crossover

The FDR noise floor $n_{\text{noise}} = \frac{1}{2} \delta_k$ crosses above the Planckian occupation at a critical frequency $\omega_{\text{cross}}$. Above this frequency, the quantum noise from dissipation dominates the thermal Hawking signal.

In [ ]:
# viz-ref: fig38

fig = go.Figure()

for name, platform in platforms.items():
    spec = compute_spectrum(platform, omega_min=0.1, omega_max_factor=8.0, n_points=100)
    T_H = platform.T_H

    omega_over_TH = spec.omega_array / T_H
    n_planck = np.array([p.n_planck for p in spec.points])
    n_noise = np.array([p.n_noise for p in spec.points])

    fig.add_trace(go.Scatter(
        x=omega_over_TH,
        y=n_planck,
        mode="lines",
        name=f"{name} n_Planck",
        line=dict(color=PLATFORM_COLORS[name], width=2),
    ))
    fig.add_trace(go.Scatter(
        x=omega_over_TH,
        y=n_noise,
        mode="lines",
        name=f"{name} n_noise",
        line=dict(color=PLATFORM_COLORS[name], width=2, dash="dash"),
    ))

apply_layout(fig,
    title="Noise Floor Crossover: Planckian vs FDR Noise",
    xaxis_title="\u03c9 / T<sub>H</sub>",
    yaxis_title="Occupation number n(\u03c9)",
)
fig.update_yaxes(type="log")
fig.show()

The dashed curves show $n_{\text{noise}}$, which grows relative to the exponentially-falling Planckian (solid curves) at high frequencies. The crossover point $\omega_{\text{cross}}$ depends on the platform's damping rate $\gamma_{\text{dim}}$: larger damping produces a lower crossover frequency and a more easily detectable noise floor.

## 7. Platform Comparison

We rank the three platforms across four key observables to guide experimental strategy.

In [ ]:
# Platform comparison
comparisons = compare_platforms(tables)

rows = []
for comp in comparisons:
    row = {"Observable": comp.observable}
    for name in ["Steinhauer", "Heidelberg", "Trento"]:
        key = f"Steinhauer_Rb87" if name == "Steinhauer" else (
              f"Heidelberg_K39" if name == "Heidelberg" else f"Trento_Na23")
        rank = comp.rankings.get(key, "--")
        val = comp.values.get(key, 0)
        row[name] = f"#{rank} ({val:.2e})"
    rows.append(row)

display(pd.DataFrame(rows))

In [ ]:
# Recommendations
for comp in comparisons:
    display(pd.DataFrame([{
        "Observable": comp.observable,
        "Recommendation": comp.recommendation,
    }]))

## 8. Measurement Strategy

For each platform, we generate a prioritized measurement strategy that accounts for detector capabilities and the relative strengths of each experiment.

In [ ]:
# Measurement strategy for each platform
for name, platform in platforms.items():
    strat = measurement_strategy(platform, tables)
    rows = []
    for i, goal in enumerate(strat.priority_measurements):
        rows.append({"Priority": i + 1, "Measurement": goal})
    display(pd.DataFrame([{"Platform": name}]))
    display(pd.DataFrame(rows))

## 9. Exact WKB vs Perturbative EFT

At sufficiently high frequencies, the exact WKB connection formula deviates from the perturbative EFT treatment by more than 1%. This crossover frequency $\omega_{\text{cross}}$ defines the regime where the exact treatment (Paper 4) provides qualitatively new predictions beyond the perturbative framework (Papers 1–2).

In [ ]:
# Exact vs perturbative comparison for each platform
for name, platform in platforms.items():
    comp = compare_exact_vs_perturbative(platform)
    T_H = platform.T_H

    crossover_str = f"{comp.crossover_omega / T_H:.2f} T_H" if comp.crossover_omega else "not found"
    display(pd.DataFrame([{
        "Platform": name,
        "Max deviation": f"{comp.max_deviation:.2e}",
        "Crossover freq": crossover_str,
    }]))

## 10. Summary and Key Results

The experimental prediction package delivers three main results:

1. **Detection feasibility**: The dissipative correction $\delta_{\text{diss}}$ is within reach of near-term improvements to existing BEC experiments. The noise floor detection requires more shots but is not ruled out.

2. **$\kappa$-scaling test**: The Heidelberg K-39 platform is uniquely positioned to perform the $\kappa$-scaling test, which cleanly separates dispersive ($\propto \kappa^2$) from dissipative ($\propto \kappa^0$) corrections.

3. **Platform-specific guidance**: Each platform has distinct advantages — Steinhauer for initial detection, Heidelberg for the scaling test, and Trento for spin-sonic enhancement of the dissipative sector.

These predictions are now ready for direct communication to experimental groups.